#  Task 08 _ AI Prototype Development

SmartCare Hospital Appointment No-Show Predictor(Streamlit)

| Requirement | Where it is done in the app |
|---|---|
| Accept user input | Patient form (age, gender, blood group, department, diagnosis, waiting days, appointment/admission history) |
| Process patient information | Same imputation → scaling → one-hot encoding → feature selection pipeline built in Task 03 |
| Generate predictions | Tuned Random Forest (Task 05/06) returns the no-show probability |
| Display results | Probability, risk level, recommended action and a SHAP explanation (Task 07) |

**How to run:** Runtime → Run all. When asked, upload `smartcare_ai_dataset_1000.csv` (from `dataset/Raw/`).

# Part 01-Install libraries

In [ ]:
!pip install -q streamlit shap

#Part 02-Upload the raw dataset

Upload `smartcare_ai_dataset_1000.csv`. The preprocessing and model are rebuilt from it with the same settings as Tasks 03 and 05 (random_state=42, 80/20 stratified split), so the prototype does not depend on pickle files from a different scikit-learn version.

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving smartcare_ai_dataset_1000.csv to smartcare_ai_dataset_1000.csv


# part 03-Pipeline module (Task 03 preprocessing + Task 05 Random Forest)

In [ ]:
%%writefile pipeline.py
"""SmartCare no-show pipeline: rebuilds the Task 03 preprocessing and the Task 05/06 Random Forest."""
import joblib
import numpy as np
import pandas as pd
import sklearn
from functools import partial
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectPercentile, mutual_info_classif
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, roc_auc_score)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler

NUM = ["age", "waiting_days", "previous_appointments",
       "missed_previous_appointments", "previous_admissions"]
CAT = ["gender", "blood_group", "department", "diagnosis"]
# Best Random Forest parameters found by GridSearchCV in Task 05
RF_PARAMS = dict(n_estimators=200, max_depth=10, min_samples_split=2,
                 min_samples_leaf=1, random_state=42)


def _dense(a):
    return a.toarray() if hasattr(a, "toarray") else np.asarray(a)


def build_bundle(csv_path):
    df = pd.read_csv(csv_path).drop_duplicates()
    df = df.dropna(subset=["no_show"])
    df = df[df["no_show"].isin([0, 1])].copy()
    X, y = df[NUM + CAT], df["no_show"].astype(int)

    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.20, random_state=42, stratify=y)

    pre = ColumnTransformer([
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                          ("scaler", RobustScaler())]), NUM),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                          ("encoder", OneHotEncoder(handle_unknown="ignore"))]), CAT),
    ])
    sel = SelectPercentile(partial(mutual_info_classif, random_state=42), percentile=80)

    A_tr = _dense(sel.fit_transform(pre.fit_transform(X_tr), y_tr))
    A_te = _dense(sel.transform(pre.transform(X_te)))
    names = pre.get_feature_names_out()[sel.get_support()]

    model = RandomForestClassifier(**RF_PARAMS).fit(A_tr, y_tr)
    pred, proba = model.predict(A_te), model.predict_proba(A_te)[:, 1]

    return {
        "preprocessor": pre, "selector": sel, "model": model,
        "feature_names": list(names),
        "X_train_selected": A_tr,
        "metrics": {"Accuracy": accuracy_score(y_te, pred),
                    "Precision": precision_score(y_te, pred),
                    "Recall": recall_score(y_te, pred),
                    "F1 Score": f1_score(y_te, pred),
                    "ROC-AUC": roc_auc_score(y_te, proba)},
        "options": {c: sorted(df[c].dropna().unique().tolist()) for c in CAT},
        "ranges": {c: (int(df[c].min()), int(df[c].max())) for c in NUM},
        "base_rate": float(y.mean()),
        "test_proba": proba,
        "sklearn_version": sklearn.__version__,
    }


def predict_patient(bundle, row: dict):
    X = pd.DataFrame([row])[NUM + CAT]
    A = _dense(bundle["selector"].transform(bundle["preprocessor"].transform(X)))
    return float(bundle["model"].predict_proba(A)[0, 1]), A


Writing pipeline.py


# Part 04-Build the model and check it matches Task 06

In [ ]:
import warnings, joblib
warnings.filterwarnings("ignore")
from pipeline import build_bundle

bundle = build_bundle("smartcare_ai_dataset_1000.csv")
joblib.dump(bundle, "model_bundle.joblib")

print("Test-set performance of the model used in the prototype:")
for k, v in bundle["metrics"].items():
    print(f"  {k:10s}: {v:.4f}")
print("\nTask 06 reported: Accuracy 0.64 | F1 0.6364 | ROC-AUC 0.6672  (should match)")

Test-set performance of the model used in the prototype:
  Accuracy  : 0.6400
  Precision : 0.6364
  Recall    : 0.6364
  F1 Score  : 0.6364
  ROC-AUC   : 0.6672

Task 06 reported: Accuracy 0.64 | F1 0.6364 | ROC-AUC 0.6672  (should match)


# Part 05-Streamlit application

In [ ]:
%%writefile app.py
"""SmartCare – Appointment No-Show Predictor (Task 08 prototype)."""
import os
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import streamlit as st

from pipeline import CAT, NUM, build_bundle, predict_patient

warnings.filterwarnings("ignore")
HERE = os.path.dirname(os.path.abspath(__file__))
CSV_NAME = "smartcare_ai_dataset_1000.csv"
CSV_CANDIDATES = [os.path.join(HERE, CSV_NAME),
                  os.path.join(HERE, "..", "dataset", "Raw", CSV_NAME),
                  os.path.join("/content", CSV_NAME)]

st.set_page_config(page_title="SmartCare No-Show Predictor", page_icon="🏥")


@st.cache_resource(show_spinner="Loading model...")
def load_bundle():
    path = os.path.join(HERE, "model_bundle.joblib")
    try:
        return joblib.load(path)
    except Exception:  # missing file or scikit-learn version mismatch -> rebuild
        csv = next(p for p in CSV_CANDIDATES if os.path.exists(p))
        return build_bundle(csv)


def risk_band(p):
    if p >= 0.60:
        return "HIGH", "error", ("Call the patient and send an SMS reminder 48 h before. "
                                 "Offer rescheduling and consider overbooking this slot.")
    if p >= 0.40:
        return "MEDIUM", "warning", "Send an SMS/WhatsApp reminder 24 h before the appointment."
    return "LOW", "success", "Standard automated reminder is sufficient."


def explain(bundle, A):
    """SHAP values for the 'no-show' class, grouped back to the original input variables."""
    import shap
    sv = shap.TreeExplainer(bundle["model"]).shap_values(A)
    sv = sv[1][0] if isinstance(sv, list) else (sv[0, :, 1] if np.ndim(sv) == 3 else sv[0])
    grouped = {}
    for name, v in zip(bundle["feature_names"], sv):
        var = next((c for c in NUM + CAT if name == f"num__{c}" or name.startswith(f"cat__{c}_")), name)
        grouped[var] = grouped.get(var, 0.0) + float(v)
    return pd.Series(grouped).sort_values(key=abs)


bundle = load_bundle()
opt, rng, m = bundle["options"], bundle["ranges"], bundle["metrics"]

# ---------------- Sidebar ----------------
with st.sidebar:
    st.header("About the model")
    st.write("Random Forest (best test F1 in Task 06) trained on the SmartCare dataset.")
    st.metric("Test accuracy", f"{m['Accuracy']:.0%}")
    st.metric("F1 score", f"{m['F1 Score']:.2f}")
    st.metric("ROC-AUC", f"{m['ROC-AUC']:.2f}")
    st.caption("Trained on a synthetic dataset – for academic demonstration only.")

# ---------------- 1. ACCEPT USER INPUT ----------------
st.title("🏥 Appointment No-Show Predictor")
st.write("Enter the patient's appointment details to estimate the chance they will miss it.")

with st.form("patient_form"):
    c1, c2 = st.columns(2)
    with c1:
        age = st.number_input("Age (years)", rng["age"][0], 100, 35)
        gender = st.selectbox("Gender", opt["gender"])
        blood_group = st.selectbox("Blood group", opt["blood_group"])
        department = st.selectbox("Department", opt["department"])
        diagnosis = st.selectbox("Diagnosis", opt["diagnosis"])
    with c2:
        waiting_days = st.slider("Waiting days (booking → appointment)", 0, 60, 10)
        prev_app = st.slider("Previous appointments", 0, 20, 3)
        missed = st.slider("Previously missed appointments", 0, 20, 0)
        prev_adm = st.slider("Previous admissions", 0, 10, 0)
    submitted = st.form_submit_button("Predict no-show risk", type="primary")

if submitted:
    if missed > prev_app:
        st.error("Missed appointments cannot exceed previous appointments.")
        st.stop()

    # ---------------- 2. PROCESS PATIENT INFORMATION ----------------
    patient = {"age": age, "gender": gender, "blood_group": blood_group,
               "department": department, "diagnosis": diagnosis,
               "waiting_days": waiting_days, "previous_appointments": prev_app,
               "missed_previous_appointments": missed, "previous_admissions": prev_adm}

    # ---------------- 3. GENERATE PREDICTION ----------------
    prob, A = predict_patient(bundle, patient)
    level, kind, action = risk_band(prob)

    # ---------------- 4. DISPLAY RESULTS ----------------
    st.subheader("Prediction result")
    st.metric("No-show probability", f"{prob:.1%}",
              delta=f"{(prob - bundle['base_rate']) * 100:+.1f} pts vs. average patient",
              delta_color="inverse")
    st.progress(min(max(prob, 0.0), 1.0))
    getattr(st, kind)(f"**{level} RISK** – {action}")

    st.subheader("Why this prediction?")
    try:
        contrib = explain(bundle, A)
        fig, ax = plt.subplots(figsize=(6, 3.6))
        ax.barh(contrib.index, contrib.values,
                color=["#d62728" if v > 0 else "#2ca02c" for v in contrib.values])
        ax.axvline(0, color="grey", lw=0.8)
        ax.set_xlabel("SHAP value  (red = raises no-show risk, green = lowers it)")
        fig.tight_layout()
        st.pyplot(fig)
    except Exception:
        imp = pd.Series(bundle["model"].feature_importances_, index=bundle["feature_names"])
        st.bar_chart(imp.sort_values().tail(8))
        st.caption("Showing global feature importance (SHAP unavailable).")

    with st.expander("Input summary"):
        st.table(pd.DataFrame(patient, index=["value"]).T.astype(str))

st.caption("⚠️ Decision-support prototype only. Predictions are probabilities, not certainties.")


Writing app.py


In [ ]:
import subprocess, time, urllib.request

subprocess.run(["pkill", "-f", "streamlit"])
log = open("streamlit.log", "w")
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"],
                 stdout=log, stderr=subprocess.STDOUT)
time.sleep(8)
print(open("streamlit.log").read())
print("Tunnel password (public IP):", urllib.request.urlopen("https://ipv4.icanhazip.com").read().decode().strip())



2026-11-20 19:57:25.498 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://104.196.113.20:8501


Tunnel password (public IP): 104.196.113.20


In [ ]:
import subprocess, re, time

subprocess.run("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared", shell=True)
log = open("cf.log", "w")
subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8501"],
                 stdout=log, stderr=subprocess.STDOUT)

for _ in range(30):
    time.sleep(2)
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open("cf.log").read())
    if m:
        print("Open this link:", m.group(0))
        break
else:
    print(open("cf.log").read()[-800:])

Open this link: https://classified-fellow-associates-bacterial.trycloudflare.com
